# Can a late search result bring a withdrawn paper back?

_Authored by: [PGG Engineering](https://huggingface.co/pgg-engineering)_

A keyword search finishes first. A vector search arrives a moment later. Then a reranker replaces the keyword results. What should the search client display after each message? Concatenating every response gives old results a second vote. Discarding the entire search also throws away a valid response from the other worker.

We'll combine ranked lists from one SciFact query, replace a worker's contribution and handle a delayed delivery after a withdrawal. First we'll rebuild the answer in plain Python. Then we'll change the arrival order 720 ways and check whether a withdrawn result can come back.

[Open in Google Colab](https://colab.research.google.com/github/pgg898200-ship-it/rankpatch/blob/main/examples/revising_hybrid_search.ipynb). Choose a CPU runtime. The reference example runs offline with the Python standard library. The final comparison installs one released wheel and works on CPython 3.11–3.13 on Linux x86-64, Apple Silicon or Windows x86-64.

## Start from three observed rankings

The lists below are the **first six document IDs** for query `1` from three published SciFact retrieval runs. BM25 supplies lexical matches, float cosine supplies vector matches, and `bm25-hamming` reranks lexical candidates using binary embeddings. The reranker operated on a deeper candidate pool, so its top six can include documents outside the BM25 top six.

We reproduce this small excerpt to make the example runnable offline. Timo Breuer published the runs with *SQLite is Enough. Lexical, Semantic, and Hybrid Search with scrydb*. The fixture retains the published order and omits raw scores, text and relevance judgments. We choose the source slots, revisions, arrival order and withdrawals for this exercise. [Original runs and license](https://huggingface.co/datasets/breuert/scrydb-eval/tree/ea5a009b6b0af4f2d164e95abca057090210db77/runs/beir) · [Extracted SciFact fixture](https://github.com/pgg898200-ship-it/rankpatch/blob/81da2bf6dc5ac41a0c0de6e0e9f942b21a261e03/replay/scifact.txt) · [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)

In [1]:
bm25 = [10608397, 18953920, 43385013, 21257564, 34386619, 40212412]
cosine = [21456232, 19855358, 36398420, 86129154, 14296612, 17388232]
reranked = [21456232, 6076903, 14492964, 17388232, 19855358, 8150638]

points = [6, 5, 4, 3, 2, 1]
k = 3

## Pick a fusion rule and a delivery rule

Give the first document on a list six points, the second five, and so on. Add each document's points across the latest list from each source. Take the three highest totals, breaking ties by smaller numeric document ID. These positional weights let us combine results without calibrating cosine similarities against BM25 scores. Changing the weights changes the policy.

Each message contains a source, its revision and its **entire current list**. Revisions increase independently for each source. Accept a message when its revision exceeds the last accepted revision for that source. An empty list withdraws the source's results. Keep its revision so an older message cannot restore them.

In [2]:
from collections import Counter


def replace_latest(latest, source, revision, documents):
    previous_revision = latest[source][0] if source in latest else -1
    if revision <= previous_revision:
        return False
    latest[source] = (revision, tuple(documents))
    return True


def rebuild(latest):
    totals = Counter()
    for _, documents in latest.values():
        for document_id, weight in zip(documents, points):
            totals[document_id] += weight
    return sorted(totals.items(), key=lambda row: (-row[1], row[0]))[:k]

## Replay the messages

The keyword worker skips revision 1 on its way to revision 2. When revision 1 eventually arrives, the withdrawal at revision 3 has already taken effect. The vector worker uses its own revision counter, starting at zero.

In [3]:
events = [
    ("keyword results",    "keyword", 0, bm25),
    ("vector results",     "vector",  0, cosine),
    ("keyword reranked",   "keyword", 2, reranked),
    ("rerank retried",     "keyword", 2, reranked),
    ("keyword withdrawn",  "keyword", 3, []),
    ("late keyword page",  "keyword", 1, bm25),
    ("vector withdrawn",   "vector",  1, []),
]

latest = {}
reference = []
for label, source, revision, documents in events:
    accepted = replace_latest(latest, source, revision, documents)
    top = rebuild(latest)
    reference.append((accepted, top))
    status = "applied" if accepted else "ignored"
    print(f"{label:18} {status:7} {top}")

keyword results    applied [(10608397, 6), (18953920, 5), (43385013, 4)]
vector results     applied [(10608397, 6), (21456232, 6), (18953920, 5)]
keyword reranked   applied [(21456232, 12), (19855358, 7), (6076903, 5)]
rerank retried     ignored [(21456232, 12), (19855358, 7), (6076903, 5)]
keyword withdrawn  applied [(21456232, 6), (19855358, 5), (36398420, 4)]
late keyword page  ignored [(21456232, 6), (19855358, 5), (36398420, 4)]
vector withdrawn   applied []


Each pair in the output is a document ID and its current score. When both original lists arrive, documents `10608397` and `21456232` tie on six points. The smaller ID wins that tie. After reranking, `21456232` is first in both live lists and scores twelve. Withdrawing the keyword list removes six of those points, and the next stale message leaves the vector ranking intact. Once both sources withdraw, the client should show an empty result.

This is enough for a service that merges once or handles just a few updates. The reference rebuild visits every active list on each read. If an application repeatedly reads the top results between small edits, it can instead remove the changed source's old points, add its new points and maintain the result incrementally.

![Two papers in the SciFact replay. The keyword leader drops from 6 to 0 after reranking. The vector leader rises from 6 to 12, stays at 12 on retry, falls to 6 after withdrawal, stays at 6 on late delivery, and finishes at 0.](https://raw.githubusercontent.com/pgg898200-ship-it/rankpatch/main/examples/assets/rankpatch-revisions.png)

## Check an incremental implementation

We'll use the open-source [RankPatch Python binding](https://github.com/pgg898200-ship-it/rankpatch/tree/v0.1.0/python/rankpatch) as a second implementation. Its index lives in one process. It expects dense integer IDs, distinct documents within each source list and no more positions than the weight table. Map the numeric SciFact IDs in ascending order to preserve the tie rule while keeping the index small. In a live service, maintain a stable mapping for the lifetime of the query.

The next cell installs the [version 0.1.0 binary release](https://github.com/pgg898200-ship-it/rankpatch/releases/tag/v0.1.0). It requires internet access for the initial download.

In [4]:
import subprocess
import sys

wheel_listing = (
    "https://github.com/pgg898200-ship-it/rankpatch/releases/expanded_assets/v0.1.0"
)
_ = subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
        "--no-index", "--only-binary=:all:", "--find-links", wheel_listing, "rankpatch==0.1.0",
    ],
    check=True,
)

In [5]:
from rankpatch import Index

document_ids = sorted(set(bm25 + cosine + reranked))
dense_id = {document_id: i for i, document_id in enumerate(document_ids)}
source_id = {"keyword": 0, "vector": 1}
index = Index(sources=len(source_id), universe=len(document_ids), rank_points=points, k=k)

for (label, source, revision, documents), expected in zip(events, reference):
    compact = [dense_id[document_id] for document_id in documents]
    accepted = index.replace(source_id[source], revision, compact)
    top = [(document_ids[i], score) for i, score in index.top()]
    assert (accepted, top) == expected, (label, expected, accepted, top)

print(f"Both implementations agree on all {len(events)} updates, including ties and withdrawals.")

Both implementations agree on all 7 updates, including ties and withdrawals.


## Deliver the same six messages in every order

Suppose the keyword worker withdraws its results while the vector worker stays online. The first six messages above contain everything the coordinator eventually receives. They can arrive in any order on a network that retries deliveries.

There are 720 permutations of the six labeled messages. The retry repeats the same payload, so some permutations are observationally identical. For each permutation, check every ranking against the plain-Python rebuild. For comparison, try a coordinator that always accepts the last message to arrive. Count how often its final ordered IDs and scores disagree with the surviving vector ranking.

In [6]:
from itertools import permutations

deliveries = events[:6]
surviving = rebuild({"vector": (0, cosine)})
permutation_count = 0
snapshot_count = 0
arrival_only_errors = 0

for order in permutations(deliveries):
    latest = {}
    arrival_only = {}
    candidate = Index(sources=len(source_id), universe=len(document_ids), rank_points=points, k=k)
    for _, source, revision, documents in order:
        accepted = replace_latest(latest, source, revision, documents)
        compact = [dense_id[doc] for doc in documents]
        assert candidate.replace(source_id[source], revision, compact) == accepted
        actual = [(document_ids[i], score) for i, score in candidate.top()]
        assert actual == rebuild(latest)
        snapshot_count += 1
        arrival_only[source] = (revision, documents)
    assert actual == surviving
    arrival_only_errors += rebuild(arrival_only) != surviving
    permutation_count += 1

print(f"Revision-aware results matched on {snapshot_count:,} intermediate snapshots.")
print(f"All {permutation_count} delivery orders ended with just the vector ranking.")
print(f"Arrival order alone gave a different final result in {arrival_only_errors} orders.")

Revision-aware results matched on 4,320 intermediate snapshots.
All 720 delivery orders ended with just the vector ranking.
Arrival order alone gave a different final result in 576 orders.


The rankings visible while messages are in flight can differ. Once the same messages have arrived, a revision-aware coordinator returns the same answer. This assumes each source assigns the same payload to a given revision. If a writer reuses a revision for two different payloads, whichever arrives first wins. The permutation counts describe this constructed exercise. They give no estimate of how often messages arrive late in production.

## Apply this to a retrieval pipeline

Keep the revision check where responses from the workers meet, before sending the combined top results to the client or reranker. A source that streams pages must accumulate them into a replacement list, or use an interface that models appends separately. Use one state object per query so concurrent searches cannot mix results.

Try changing the rerank retry's payload while leaving its revision at 2. Both implementations should still ignore it. Then change the weights and rerun the notebook. Agreement checks the implementation of your scoring rule. To choose weights for search quality, evaluate against held-out relevance judgments. This replay checks ordering and update semantics.

To explore how an outage changes the context passed to a reader across 16 queries, open the [retriever outage notebook](https://github.com/pgg898200-ship-it/rankpatch/blob/main/examples/retriever_outage.ipynb).